In [4]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os, sys, shutil, importlib
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "rebuttal27").is_dir():
    ROOT = ROOT / "AAAI27_rebuttal_verified"
if not (ROOT / "rebuttal27").is_dir():
    raise FileNotFoundError("Notebook must run from AAAI27_rebuttal_verified or its parent")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import rebuttal27.runner as runner
assert Path(runner.__file__).resolve() == (ROOT / "rebuttal27" / "runner.py").resolve(), "Restart Kernel: old runner loaded"
importlib.reload(runner)
runner.verify_untouched_sources()
print("Verified original source SHA256; project:", ROOT)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Verified original source SHA256; project: E:\positron\NFlow\AAAI27_rebuttal_verified


In [5]:
TASK = "depth"  # depth / mfvb / difficult / correlation / poisson / boston / gse
SEED = 400
PROFILE = "pilot"
USE_MCMC = True
RSCRIPT = r"C:\Program Files\R\R-4.5.1\bin\Rscript.exe"
MCMC_N = 10000  # 原本 MCMC 采样预算保留；独立于 Table 2 的 VI 配置
MCMC_BURN = 2000
MCMC_SMAX = 100
BETA_EPS = 0.05  # 与用户已完成的 ESS 运行一致
THREADS = 8
MFVB_EPOCHS = 600
MFVB_SLAB_VAR = 0.5  # EM 的初始值，非固定方差
MFVB_SLAB_MODE = "em"
MFVB_RESTARTS = 3
COMMON = dict(seed=SEED, profile=PROFILE, mcmc=USE_MCMC, rscript=RSCRIPT,
              mcmc_n=MCMC_N, mcmc_burn=MCMC_BURN, mcmc_smax=MCMC_SMAX,
              beta_eps=BETA_EPS, threads=THREADS, mfvb_epochs=MFVB_EPOCHS,
              mfvb_slab_var=MFVB_SLAB_VAR, mfvb_slab_mode=MFVB_SLAB_MODE,
              mfvb_restarts=MFVB_RESTARTS)
if USE_MCMC and not (shutil.which(RSCRIPT) or Path(RSCRIPT).is_file()):
    raise FileNotFoundError("Rscript not found. Set RSCRIPT to the installed Rscript.exe")
print("Task:", TASK, "| profile:", PROFILE, "| default paper depth: 64+4")


Task: depth | profile: pilot | default paper depth: 64+4


## 1. Flow depth ablation（所有 depth 使用所选 PROFILE 的训练日程）


In [6]:
if TASK == 'depth':
    result = runner.run_depth(depths=(4,8,16,32), setting='baseline', **COMMON)
    display(result)

[MCMC] fresh original R ESS: gaussian N=10000 burn=2000 beta_eps=0.05
[run] baseline rat depth=4 seed=400 profile=pilot
[stage 00 | epoch 0001] tau=0.5000 lr=2.00e-04 loss=1628.843506 score=11.5923 S= 35 pip=48.58 churn=0.000 ok*
[stage 00 | epoch 0050] tau=0.5000 lr=2.00e-04 loss=795.882141 score=8.4035 S= 94 pip=92.86 churn=0.095 ok*
[stage 00 | epoch 0100] tau=0.5000 lr=2.00e-04 loss=384.662140 score=3.3802 S= 43 pip=46.59 churn=0.348 ok*
[stage 01 | epoch 0101] tau=0.3824 lr=2.00e-04 loss=390.335327 score=3.3816 S= 47 pip=47.66 churn=0.163 ok*
[stage 01 | epoch 0150] tau=0.3824 lr=2.00e-04 loss=324.063324 score=2.3581 S= 18 pip=34.33 churn=0.400 ok*
[stage 01 | epoch 0200] tau=0.3824 lr=2.00e-04 loss=283.399109 score=1.7737 S=  9 pip=14.25 churn=0.182 ok*
[stage 02 | epoch 0201] tau=0.2924 lr=1.80e-04 loss=287.520966 score=1.6782 S=  9 pip=14.05 churn=0.000 ok*
[stage 02 | epoch 0250] tau=0.2924 lr=1.80e-04 loss=254.266876 score=1.6136 S=  9 pip=9.95 churn=0.000 ok*
[stage 02 | epo

,dataset,family,seed,method,depth,n_coupling_transforms,coefficient_map,reference_coefficient_map,same_posterior_target,profile,...,selected_epoch,selected_tau,cross_prior_effective_PIP_RMSE,cross_prior_beta_mean_RMSE,cross_prior_joint_inclusion_MAE,cross_prior_active_marginal_SKL,cross_prior_zero_JS,cross_prior_active_pairwise_SKL,cross_prior_active_joint_SW2,MCMC_seconds
0,baseline,gaussian,400,MCMC_original_R_ESS,0,0,relu,relu,True,original_R,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,baseline,gaussian,400,rat,4,12,sigmoid_train_indicator_report,relu,False,pilot,...,401.0,0.170998,0.122899,0.035417,0.125663,0.329678,0.021462,0.663527,0.106753,98.806293
2,baseline,gaussian,400,rat,8,24,sigmoid_train_indicator_report,relu,False,pilot,...,501.0,0.130766,0.107253,0.026974,0.020848,0.165759,0.021689,0.300609,0.059636,98.806293
3,baseline,gaussian,400,rat,16,48,sigmoid_train_indicator_report,relu,False,pilot,...,301.0,0.223607,0.128492,0.044896,0.144515,0.495948,0.021326,0.994805,0.128590,98.806293
4,baseline,gaussian,400,rat,32,96,sigmoid_train_indicator_report,relu,False,pilot,...,501.0,0.130766,0.108349,0.031282,0.048626,0.239726,0.020461,0.442009,0.080308,98.806293


## 2. Traditional point-mass spike-and-slab MFVB


In [7]:
TASK = 'mfvb'
if TASK == 'mfvb':
    result = runner.run_mfvb(setting='baseline', family='gaussian', depth=68, **COMMON)
    display(result)

[MCMC] fresh original R ESS: gaussian N=10000 burn=2000 beta_eps=0.05
[run] baseline mfvb depth=0 seed=400 profile=pilot
[result] baseline mfvb K=0 time=0.17s support=10 test_nll=1.344
[run] baseline rat depth=68 seed=400 profile=pilot
[stage 00 | epoch 0001] tau=0.5000 lr=2.00e-04 loss=1619.869263 score=11.4082 S=  1 pip=26.17 churn=0.000 ok*
[stage 00 | epoch 0050] tau=0.5000 lr=2.00e-04 loss=297.896606 score=2.3925 S= 10 pip=15.39 churn=0.667 ok*
[stage 00 | epoch 0100] tau=0.5000 lr=2.00e-04 loss=252.791382 score=1.7964 S=  9 pip=9.17 churn=0.000 ok*
[stage 01 | epoch 0101] tau=0.3824 lr=2.00e-04 loss=250.058517 score=1.6472 S=  9 pip=8.97 churn=0.000 ok*
[stage 01 | epoch 0150] tau=0.3824 lr=2.00e-04 loss=243.368225 score=1.6456 S=  9 pip=9.19 churn=0.000 ok
[stage 01 | epoch 0200] tau=0.3824 lr=2.00e-04 loss=241.606155 score=1.5577 S=  9 pip=9.17 churn=0.000 ok*
[stage 02 | epoch 0201] tau=0.2924 lr=1.80e-04 loss=240.765808 score=1.5462 S=  9 pip=9.38 churn=0.000 ok*
[stage 02 | 

,dataset,family,seed,method,depth,n_coupling_transforms,coefficient_map,reference_coefficient_map,same_posterior_target,profile,...,cross_prior_beta_mean_RMSE,cross_prior_joint_inclusion_MAE,cross_prior_active_marginal_SKL,cross_prior_zero_JS,cross_prior_active_pairwise_SKL,cross_prior_active_joint_SW2,MCMC_seconds,training_epochs,selected_epoch,selected_tau
0,baseline,gaussian,400,MCMC_original_R_ESS,0,0,relu,relu,True,original_R,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,baseline,gaussian,400,mfvb,0,0,pointmass_gaussian,relu,False,pilot,...,0.018365,0.012967,0.057775,0.013106,0.114923,0.035393,122.323678,NaN,NaN,NaN
2,baseline,gaussian,400,rat,68,204,sigmoid_train_indicator_report,relu,False,pilot,...,0.023545,0.032589,0.091341,0.020357,0.180764,0.053559,122.323678,700.0,601.0,0.1


## 3. Difficult Gaussian: MFVB / RAT / Odd-even / Thinner


In [ ]:
if TASK == 'difficult':
    result = runner.run_difficult(settings=('weak_signal','low_snr','p_gt_n'), depth=68, **COMMON)
    display(result)

## 4. Correlated predictors（AR(1)）


In [ ]:
if TASK == 'correlation':
    result = runner.run_correlation(rhos=(.25,.50,.75), depth=68, **COMMON)
    display(result)

## 5. Poisson count regression


In [ ]:
if TASK == 'poisson':
    result = runner.run_poisson(settings=('baseline','weak_signal','low_snr','n_gt_p','p_gt_n'), depth=68, **COMMON)
    display(result)

## 6. Boston（Gaussian）


In [ ]:
if TASK == 'boston':
    result = runner.run_boston(n_noise=12, **COMMON)
    display(result)

## 7. GSE32448（microarray；tumor vs benign/normal logistic）


In [ ]:
if TASK == 'gse':
    result = runner.run_gse(n_genes=40, n_noise=30, outcome='tumor', **COMMON)
    display(result)

## 结果说明

一个 task 输出一个 CSV。SKL 不可估计的分量跳过，报告成功估计项的中位数，不再输出 `valid/total` 列；全部不可估计时保持 NaN。
原始 R ESS 的缓存按 likelihood family 隔离。跨先验的 posterior metrics 使用 `cross_prior_*` 前缀。


In [ ]:
if 'result' in globals():
    show = ['dataset','method','depth','n_coupling_transforms','wall_seconds',
            'support_size','TPR','FPR','test_nll','test_mse',
            'noise_FPR','cross_prior_effective_PIP_RMSE']
    display(result[[k for k in show if k in result.columns]])